<a href="https://colab.research.google.com/github/JaydenTran04/AAI2026/blob/main/Part_1_Predict_house_prices_based_on_square_footage_and_location_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import files
uploaded = files.upload()


Saving housing_train.csv to housing_train.csv


In [13]:
import pandas as pd, os, numpy as np, random

in_path = "housing_train.csv"
df = pd.read_csv(in_path)

df.head(), df.columns, df.shape

(   city_index  bedrooms  bathrooms         sqft     lot_size  year_built  \
 0           0         1          2  1790.532648  5040.262683        1984   
 1           3         1          1  1070.847921  4903.369554        2020   
 2           3         4          3  1698.109778  4535.351578        1961   
 3           2         4          3  1237.794943  2603.420686        1968   
 4           2         4          2  1633.557849  7996.628975        1986   
 
    distance_to_center  school_rating  crime_rate  hoa_fee         price  
 0            8.915195       6.015530   38.976337      100  6.818944e+05  
 1            4.072055       5.813620   43.809815        0  7.841239e+05  
 2            4.378840       3.665797   24.856335      200  9.779437e+05  
 3           13.032852       6.987031   26.516467      100  1.115974e+06  
 4            2.010885       7.767718   54.793759      100  1.238674e+06  ,
 Index(['city_index', 'bedrooms', 'bathrooms', 'sqft', 'lot_size', 'year_built',
    

In [15]:
import pandas as pd, numpy as np, random, os

in_path = "housing_train.csv"

df = pd.read_csv(in_path)

# ---------- Prompt 1 ----------
clean = df.loc[:, ["price", "sqft"]].rename(columns={"sqft": "footage"})

out_csv1 = "housing_price_footage.csv"
out_xlsx1 = "housing_price_footage.xlsx"

clean.to_csv(out_csv1, index=False)
clean.to_excel(out_xlsx1, index=False)

# ---------- Prompt 2 ----------
rng = np.random.default_rng(42)
locations = np.array(["Downtown", "Rural", "Suburb"])

clean2 = clean.copy()
clean2["location"] = rng.choice(locations, size=len(clean2), replace=True)

out_csv2 = "housing_price_footage_with_location.csv"
out_xlsx2 = "housing_price_footage_with_location.xlsx"

clean2.to_csv(out_csv2, index=False)
clean2.to_excel(out_xlsx2, index=False)

clean.head(), clean2.head(), (out_csv1, out_csv2, out_xlsx1, out_xlsx2)

(          price      footage
 0  6.818944e+05  1790.532648
 1  7.841239e+05  1070.847921
 2  9.779437e+05  1698.109778
 3  1.115974e+06  1237.794943
 4  1.238674e+06  1633.557849,
           price      footage  location
 0  6.818944e+05  1790.532648  Downtown
 1  7.841239e+05  1070.847921    Suburb
 2  9.779437e+05  1698.109778     Rural
 3  1.115974e+06  1237.794943     Rural
 4  1.238674e+06  1633.557849     Rural,
 ('housing_price_footage.csv',
  'housing_price_footage_with_location.csv',
  'housing_price_footage.xlsx',
  'housing_price_footage_with_location.xlsx'))

In [16]:
import pandas as pd

path = "housing_price_footage_with_location.xlsx"
df = pd.read_excel(path)

# quick check
print(df.shape)
print(df.columns)
df.head()

(5000, 3)
Index(['price', 'footage', 'location'], dtype='object')


,price,footage,location
0,6.818944e+05,1790.532648,Downtown
1,7.841239e+05,1070.847921,Suburb
2,9.779437e+05,1698.109778,Rural
3,1.115974e+06,1237.794943,Rural
4,1.238674e+06,1633.557849,Rural


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

X = df[["footage", "location"]]
y = df["price"]

# OneHotEncoder for the categorical column
preprocess = ColumnTransformer(
    transformers=[
        ("loc", OneHotEncoder(drop="first", handle_unknown="ignore"), ["location"])
    ],
    remainder="passthrough"  # keeps "footage" as a numeric feature
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", LinearRegression())
])

model.fit(X, y)
print("Model trained.")

Model trained.


In [18]:
sample = pd.DataFrame([{"footage": 2000, "location": "Downtown"}])
pred = model.predict(sample)[0]
print("Predicted price for 2000 sqft in Downtown:", pred)

Predicted price for 2000 sqft in Downtown: 848933.3059949352


In [19]:
# Get feature names after encoding
ohe = model.named_steps["preprocess"].named_transformers_["loc"]
loc_feature_names = ohe.get_feature_names_out(["location"])  # e.g., location_Rural, location_Suburb (depends on drop)

# Because remainder="passthrough", footage is appended after the one-hot columns
feature_names = list(loc_feature_names) + ["footage"]

coefs = model.named_steps["reg"].coef_
intercept = model.named_steps["reg"].intercept_

print("Intercept:", intercept)
for name, coef in zip(feature_names, coefs):
    print(f"{name}: {coef}")

print("\nPer-square-foot increase (footage coefficient):", coefs[-1])
print("Location effects are the location_* coefficients (relative to the dropped baseline).")

Intercept: 346107.8019708296
location_Rural: 378566.52541747957
location_Suburb: 110614.79836470487
footage: 251.4127520120528

Per-square-foot increase (footage coefficient): 251.4127520120528
Location effects are the location_* coefficients (relative to the dropped baseline).


In [ ]:
1. The dataset contains three variables: price (target), footage (numeric predictor), and location (categorical).
2. OneHotEncoder, which converts it into numeric indicator variables (e.g., location_Rural, location_Suburb). One category is dropped and serves as the baseline.
3. A linear regression model is trained using both square footage and location.
4. Linear regression model is trained using scikit-learn
5. I predicted the price of a 2000 sq ft house in Downtown
6. The model coefficients show:

The per-square-foot price increase (coefficient on footage).

The location effects, which represent how much Rural or Suburb homes differ in price compared with the baseline location.

7. Equation: predicted price = intercept + (footage coef × footage)+(location effect)